<a href="https://colab.research.google.com/github/Ivan137950/Samokat_kaggle/blob/main/samokat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [760]:
!pip install catboost
!pip install optuna

import catboost as ctb
from catboost import CatBoostRegressor

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Set, Iterable
import optuna


# Preprocessing

## Метод пристального взгляда

*   `shifts_prediction.csv` - прогноз числа заказов на неделю и запланированная нагрузка (нормативное значение числа заказов на одного курьера в час), а также прогнозное число курьеров, чтобы обслужить эти заказы. Каждая дата - понедельник соответствующей недели, на которую приведены прогнозные значения.

*   `facts.csv` - файл содержит реальное кол-во курьеров, которое нужно было, чтобы обслужить все фактические заказы за неделю, предшествующую текущей (маркером текущей недели является также понедельник). Также содержит фактическую нагрузку за прошедшую неделю, число курьеров, предлагавших свои услуги (не обязательно вышедших на смены), фактическое кол-во заказов, лайфтайм даркстора, название города, общее число трудоустроенных курьеров, число курьеров, вышедших на смены, расходы на маркетинг, факт повышенного спроса.

*   `train.csv` - содержит для каждого даркстора на даты в прошлом дефицит курьеров в поле target. В файле, очевидно, не указаны значения для тестовой даты, на которую необходимо сделать прогноз!

*   `test.csv` - тестовый датасет, упорядочен по store_id. Необходимо сделать прогноз для каждого из приведенных в файле дарксторов (на единственную дату - неделю, следующую за последней известной в train.csv, т.е. на 24 ноября 2025 г.). Сама дата, на которую делается прогноз опущена в этом файле. Признаки для этой даты для каждого даркстора за соответствующую дату можно найти в приведенных выше файлах facts.csv, shifts_prediction.csv.

*   `sample_submission.csv` - пример файла для отправки решения.

In [809]:
facts = pd.read_csv("/content/facts.csv")
# sample_submission = pd.read_csv("/content/sample_submission.csv")
shifts_prediction = pd.read_csv("/content/shifts_prediction.csv")
test = pd.read_csv("/content/test.csv")
train = pd.read_csv("/content/train.csv")

**shifts_prediction.csv:**

*   `calendar_dt` - дата в формате YYYY-MM-DD
*   `store_id` - идентификатор даркстора
*   `predicted_staff_value` - предсказанное число человек, необходимое, чтобы обслужить все заказы на неделе прогноза,
*   `predicted_num_orders` - предсказанное число заказов на неделю прогноза,
*   `predicted_load_factor` - предсказанная нагрузка на курьеров (число заказов в час, которое приходится на одного курьера) на неделю прогноза.

**facts.csv:**

*   `calendar_dt` - дата в формате YYYY-MM-DD
*   `store_id` - идентификатор даркстора
*   `fact_staff_value_lag_1` - фактическое число человек, необходимое, чтобы обслужить все заказы на прошлой неделе,
*   `fact_load_factor_lag_1` - фактическая нагрузка на курьеров на прошлой неделе,
*   `num_available_couriers_lag_1` - число курьеров, предложивших свои услуги за предыдущую неделю,
*   `fact_num_orders_lag_1` - фактическое число заказов за предыдущую неделю,
*   `fact_percent_lateness_lag_1` - процент опозданий за предыдущую неделю,
*   `store_lifetime_in_days` - число дней функционирования даркстора,
*   `city_nm` - город функционирования даркстора,
*   `fact_staff_churn` - число курьеров, уволившихся из компании,
*   `flag_high_load_lag_1` - признак повышенного спроса за предыдущую неделю,
*   `marketing_costs_lag_1` - затраты на маркетинг, осуществленные за предыдущую неделю,
*   `fact_couriers_with_shifts_lag_1` - число курьеров, выходивших на смены на предыдущей неделе.

In [810]:
facts.sort_values(
    "calendar_dt",
    axis=0,
    ascending=False,
    inplace=False,
    kind='quicksort',
    na_position='last',
    ignore_index=False,
    key=None
    ).head()

,calendar_dt,store_id,fact_staff_value_lag_1,fact_load_factor_lag_1,num_available_couriers_lag_1,fact_num_orders_lag_1,fact_percent_lateness_lag_1,city_nm,store_lifetime_in_days,fact_staff_churn,flag_high_load_lag_1,marketing_costs_lag_1,fact_couriers_with_shifts_lag_1
10659,2025-11-24,ffd9283f-50d7-11ef-b973-08c0eb32008b,11,3.428571,15,48,95.238095,Самара,474.0,0.0,0,6.993008e+05,14.0
9246,2025-11-24,de6a946e-f002-11ee-b10b-08c0eb31fffb,9,2.153846,8,28,100.000000,Рязань,543.0,2.0,1,3.184646e+06,13.0
3871,2025-11-24,560c39d5-b6f3-11eb-85ab-1c34dae33151,7,2.900000,11,29,100.000000,Санкт-Петербург,1641.0,1.0,1,8.537551e+08,10.0
3875,2025-11-24,56210021-ec5e-11ee-b10b-08c0eb31fffb,11,3.833333,15,46,66.666667,Москва,581.0,0.0,0,2.786011e+07,12.0
9254,2025-11-24,deafcece-dbad-11ee-b10b-08c0eb31fffb,12,4.100000,12,41,100.000000,Москва,604.0,5.0,1,2.213916e+08,10.0


In [811]:
shifts_prediction.head(10)

,calendar_dt,store_id,predicted_staff_value,predicted_num_orders,predicted_load_factor
0,2024-01-01,000fade4-e8dc-11ed-b10a-08c0eb31fffb,12,270,2.85
1,2024-01-08,000fade4-e8dc-11ed-b10a-08c0eb31fffb,14,310,2.85
2,2024-01-15,000fade4-e8dc-11ed-b10a-08c0eb31fffb,15,370,2.96
3,2024-01-22,000fade4-e8dc-11ed-b10a-08c0eb31fffb,14,350,2.95
4,2024-01-29,000fade4-e8dc-11ed-b10a-08c0eb31fffb,15,350,2.75
5,2024-02-05,000fade4-e8dc-11ed-b10a-08c0eb31fffb,15,340,2.75
6,2024-02-12,000fade4-e8dc-11ed-b10a-08c0eb31fffb,13,330,2.75
7,2024-03-04,000fade4-e8dc-11ed-b10a-08c0eb31fffb,13,420,3.20
8,2024-03-11,000fade4-e8dc-11ed-b10a-08c0eb31fffb,13,360,3.20
9,2024-03-18,000fade4-e8dc-11ed-b10a-08c0eb31fffb,13,370,3.30


In [812]:
print(360 / 13 /  8)

3.4615384615384617


In [813]:
train.sort_values(
    "calendar_dt",
    axis=0,
    ascending=True,
    inplace=False,
    kind='quicksort',
    na_position='last',
    ignore_index=False,
    key=None
    ).head()

,calendar_dt,store_id,target
6588,2024-01-01,cf69cefd-2016-11eb-8599-1c34dae33151,14.0
3367,2024-01-01,60548799-a410-11eb-85a9-1c34dae33151,14.0
7744,2024-01-01,f46701f1-af2c-11ea-b969-0050560306e1,8.0
653,2024-01-01,12ff0515-9aa3-11ea-bc7d-0050560306e1,9.0
6589,2024-01-08,cf69cefd-2016-11eb-8599-1c34dae33151,19.0


In [814]:
test.head()

,store_id
0,000fade4-e8dc-11ed-b10a-08c0eb31fffb
1,0022f1b0-b8f8-11ee-b10b-08c0eb31fffb
2,00440ac1-6a1d-11eb-85a3-1c34dae33151
3,00442959-9671-11ec-ae6d-08c0eb320147
4,00562194-569a-11ec-a0ee-ec0d9a21b021


In [815]:
print(f"facts.shape: {facts.shape}")
print(f"shifts_prediction.shape: {shifts_prediction.shape}")
print(f"test.shape: {test.shape}")
print(f"train.shape: {train.shape}")

facts.shape: (10660, 13)
shifts_prediction.shape: (223470, 5)
test.shape: (2438, 1)
train.shape: (8220, 3)


## Shifts_prediction delete exceptions

In [816]:
'''
predicted_load_factor -- предсказанная нагрузка на курьеров (число заказов в час, которое приходится на одного курьера) на неделю прогноза.

predicted_load_factor
count	2.232710e+05
mean	4.062309e+03
std	1.355652e+06
min	6.300000e-01
25%	3.700000e+00
50%	4.100000e+00
75%	5.000000e+00
max	4.529500e+08 <-- Курьер столько не осилит будто бы...
Стоит разобраться с данной колонкой отдельно
'''
shifts_prediction.describe()


,predicted_staff_value,predicted_num_orders,predicted_load_factor
count,223470.000000,223470.000000,2.232710e+05
mean,9.791628,335.561239,4.062309e+03
std,5.087184,156.167285,1.355652e+06
min,0.000000,0.000000,6.300000e-01
25%,6.000000,230.000000,3.700000e+00
50%,9.000000,310.000000,4.100000e+00
75%,12.000000,420.000000,5.000000e+00
max,58.000000,3170.000000,4.529500e+08


In [817]:
shifts_prediction.isnull().sum()

,0
calendar_dt,0
store_id,0
predicted_staff_value,0
predicted_num_orders,0
predicted_load_factor,199


In [818]:
exceptions = shifts_prediction[
    shifts_prediction['predicted_load_factor'] > 15
    ].sort_values('predicted_load_factor', ascending=True).copy()
exceptions

,calendar_dt,store_id,predicted_staff_value,predicted_num_orders,predicted_load_factor
106518,2025-07-28,7389cb2b-a946-11ea-9e08-0050560306e1,3,190,100000.0
7953,2024-08-05,09302183-851f-11ee-b10b-08c0eb31fffb,6,190,452950000.0
58773,2024-08-05,3e94ef9b-15a0-11ee-ae78-08c0eb320147,14,450,452950000.0


In [819]:
shifts_prediction[(shifts_prediction['predicted_load_factor'].isnull()) & (shifts_prediction['predicted_num_orders'] > 0)]
# т.е.  (число заказов в час, которое приходится на одного курьера) IsNan <=> предсказанное число заказов на неделю прогноза == 0.
#  Вероятно, Nan можно заменить на 0 (число заказов на курьера == 0 когда предсказанное число заказов на неделю прогноза == 0)

,calendar_dt,store_id,predicted_staff_value,predicted_num_orders,predicted_load_factor


In [820]:
shifts_prediction.loc[shifts_prediction['predicted_load_factor'].isnull(), 'predicted_load_factor'] = 0

In [821]:
NORMAL_LOAD_FACTOR_THRESHOLD = 15

mean_normal_load_factors = shifts_prediction[
    shifts_prediction['predicted_load_factor'] <= NORMAL_LOAD_FACTOR_THRESHOLD
].groupby(['predicted_staff_value', 'predicted_num_orders'])['predicted_load_factor'].mean()

indices_to_update = shifts_prediction[shifts_prediction['predicted_load_factor'] > NORMAL_LOAD_FACTOR_THRESHOLD].index

exception_keys = shifts_prediction.loc[indices_to_update, ['predicted_staff_value', 'predicted_num_orders']]
mapped_replacements = exception_keys.set_index([
    'predicted_staff_value',
    'predicted_num_orders'
    ]).index.map(mean_normal_load_factors)

# Если для какой-то комбинации (predicted_staff_value, predicted_num_orders) не нашлось нормального среднего,
# заменяем на общее среднее всех нормальных значений predicted_load_factor.
overall_mean_normal = shifts_prediction[shifts_prediction['predicted_load_factor'] <= NORMAL_LOAD_FACTOR_THRESHOLD]['predicted_load_factor'].mean()
mapped_replacements = mapped_replacements.fillna(overall_mean_normal)

shifts_prediction.loc[indices_to_update, 'predicted_load_factor'] = mapped_replacements.values

### Результат

In [822]:
shifts_prediction.describe()

,predicted_staff_value,predicted_num_orders,predicted_load_factor
count,223470.000000,223470.000000,223470.000000
mean,9.791628,335.561239,4.456255
std,5.087184,156.167285,1.207723
min,0.000000,0.000000,0.000000
25%,6.000000,230.000000,3.700000
50%,9.000000,310.000000,4.100000
75%,12.000000,420.000000,5.000000
max,58.000000,3170.000000,12.850000


In [823]:
shifts_prediction.isnull().sum()

,0
calendar_dt,0
store_id,0
predicted_staff_value,0
predicted_num_orders,0
predicted_load_factor,0


## Facts prepr delete exceptions

In [824]:
facts.describe()

,fact_staff_value_lag_1,fact_load_factor_lag_1,num_available_couriers_lag_1,fact_num_orders_lag_1,fact_percent_lateness_lag_1,store_lifetime_in_days,fact_staff_churn,flag_high_load_lag_1,marketing_costs_lag_1,fact_couriers_with_shifts_lag_1
count,10660.000000,10543.000000,10660.000000,10660.000000,8201.000000,10660.000000,10660.000000,10660.000000,7.457000e+03,10543.000000
mean,7.206660,2.929684,10.861538,33.038743,81.179845,976.580769,1.853002,0.677580,4.140296e+10,12.689367
std,4.647654,1.969138,3.948154,18.533299,19.055068,584.030709,1.860373,0.467425,8.297097e+11,5.145462
min,1.000000,0.000000,0.000000,0.000000,2.702703,0.000000,0.000000,0.000000,7.559291e+03,1.000000
25%,4.000000,1.636364,9.000000,17.000000,71.428571,500.000000,0.000000,0.000000,6.126757e+07,9.000000
50%,7.000000,2.642857,10.000000,33.000000,85.714286,909.000000,1.000000,1.000000,4.499068e+08,12.000000
75%,10.000000,3.750000,12.000000,44.000000,100.000000,1450.000000,3.000000,1.000000,3.649713e+09,15.000000
max,40.000000,45.000000,44.000000,215.000000,100.000000,2527.000000,5.000000,1.000000,6.283868e+13,40.000000


In [825]:
facts.isnull().sum()


,0
calendar_dt,0
store_id,0
fact_staff_value_lag_1,0
fact_load_factor_lag_1,117
num_available_couriers_lag_1,0
fact_num_orders_lag_1,0
fact_percent_lateness_lag_1,2459
city_nm,0
store_lifetime_in_days,0
fact_staff_churn,0



* fact_percent_lateness_lag_1 - процент опозданий за предыдущую неделю
* fact_load_factor_lag_1 - фактическая нагрузка на курьеров на прошлой неделе
* marketing_costs_lag_1 - затраты на маркетинг, осуществленные за предыдущую неделю
* fact_couriers_with_shifts_lag_1 - число курьеров, выходивших на смены на предыдущей неделе
---

Все вышеперечисенные параметры будут заменены среднее значение по городу за аналогичный период

In [826]:
# Преобразование 'calendar_dt' в формат даты и создание
facts['calendar_dt'] = pd.to_datetime(facts['calendar_dt'])
facts['week'] = facts['calendar_dt'].dt.isocalendar().week
facts['year'] = facts['calendar_dt'].dt.isocalendar().year

# Колонки для заполнения пропусков
columns_to_impute = [
    'fact_percent_lateness_lag_1',
    'fact_load_factor_lag_1',
    'marketing_costs_lag_1',
    'fact_couriers_with_shifts_lag_1'
]

for col in columns_to_impute:
    city_week_mean = facts.groupby(['city_nm', 'calendar_dt'])[col].transform('mean')
    facts[col] = facts[col].fillna(city_week_mean)
    facts[col] = facts[col].fillna(facts[col].mean()) # Случай, когда для комбинации город-дата все значения NaN

print("Количество NaN в 'facts' после заполнения пропусков:")
print(facts[columns_to_impute].isnull().sum())

Количество NaN в 'facts' после заполнения пропусков:
fact_percent_lateness_lag_1        0
fact_load_factor_lag_1             0
marketing_costs_lag_1              0
fact_couriers_with_shifts_lag_1    0
dtype: int64


### Результат

In [827]:
facts.describe()


,calendar_dt,fact_staff_value_lag_1,fact_load_factor_lag_1,num_available_couriers_lag_1,fact_num_orders_lag_1,fact_percent_lateness_lag_1,store_lifetime_in_days,fact_staff_churn,flag_high_load_lag_1,marketing_costs_lag_1,fact_couriers_with_shifts_lag_1,week,year
count,10660,10660.000000,10660.000000,10660.000000,10660.000000,10660.000000,10660.000000,10660.000000,10660.000000,1.066000e+04,10660.000000,10660.0,10660.0
mean,2025-10-19 20:38:35.347092224,7.206660,2.932128,10.861538,33.038743,81.183101,976.580769,1.853002,0.677580,4.239093e+10,12.692047,44.770263,2024.965572
min,2024-01-01 00:00:00,1.000000,0.000000,0.000000,0.000000,2.702703,0.000000,0.000000,0.000000,7.559291e+03,1.000000,1.0,2024.0
25%,2025-11-03 00:00:00,4.000000,1.642857,9.000000,17.000000,76.190476,500.000000,0.000000,0.000000,1.450233e+08,9.000000,45.0,2025.0
50%,2025-11-10 00:00:00,7.000000,2.666667,10.000000,33.000000,81.183101,909.000000,1.000000,1.000000,1.844237e+09,12.000000,46.0,2025.0
75%,2025-11-17 00:00:00,10.000000,3.750000,12.000000,44.000000,93.983957,1450.000000,3.000000,1.000000,1.462001e+10,15.000000,47.0,2025.0
max,2025-11-24 00:00:00,40.000000,45.000000,44.000000,215.000000,100.000000,2527.000000,5.000000,1.000000,6.283868e+13,40.000000,52.0,2025.0
std,NaN,4.647654,1.976759,3.948154,18.533299,16.717575,584.030709,1.860373,0.467425,7.023697e+11,5.125039,7.003399,0.182334


In [828]:
facts.isnull().sum()

,0
calendar_dt,0
store_id,0
fact_staff_value_lag_1,0
fact_load_factor_lag_1,0
num_available_couriers_lag_1,0
fact_num_orders_lag_1,0
fact_percent_lateness_lag_1,0
city_nm,0
store_lifetime_in_days,0
fact_staff_churn,0


# Make final dataset

Осталось
* Добавить среднее по городу (в качестве доп фичей) по следующим столбцам:

    `fact_staff_value_lag_1`,
    `fact_load_factor_lag_1`,
    `num_available_couriers_lag_1`,
    `fact_num_orders_lag_1`,
    `fact_percent_lateness_lag_1`,
    `marketing_costs_lag_1`,
    `fact_couriers_with_shifts_lag_1`,
    `predicted_staff_value`,
    `predicted_num_orders`,
    `predicted_load_factor`
* Объединить имеющиеся таблицы
* Добавить таргеты

---

---


---


---


In [830]:
echo "# Samokat_kaggle" >> README.md
git init
git add README.md
git commit -m "first commit"
git branch -M main
git remote add origin https://github.com/Ivan137950/Samokat_kaggle.git
git push -u origin main

UsageError: Line magic function `%git` not found.


# Буду переделывать

In [ ]:
#Каждый раз дата обновляется в понедельник, дату можно заменить на номер недели + года
facts['calendar_dt'] = pd.to_datetime(facts['calendar_dt'])
facts['week'] = facts['calendar_dt'].dt.isocalendar().week
facts['year'] = facts['calendar_dt'].dt.isocalendar().year
first_year = list(facts.sort_values('year', ascending=True)['year'])[0]
facts["weeks_past"] = (facts["year"] - first_year) * 52 + facts["week"]

train['calendar_dt'] = pd.to_datetime(train['calendar_dt'])
train['week'] = train['calendar_dt'].dt.isocalendar().week
train['year'] = train['calendar_dt'].dt.isocalendar().year
train["weeks_past"] = (train["year"] - first_year) * 52 + train["week"]

shifts_prediction['calendar_dt'] = pd.to_datetime(shifts_prediction['calendar_dt'])
shifts_prediction['week'] = shifts_prediction['calendar_dt'].dt.isocalendar().week
shifts_prediction['year'] = shifts_prediction['calendar_dt'].dt.isocalendar().year
shifts_prediction["weeks_past"] = (shifts_prediction["year"] - first_year) * 52 + shifts_prediction["week"]

In [ ]:
last_train_year = list(train.sort_values('year', ascending=False)['year'])[0]
last_train_week = list(train[train['year'] >= last_train_year].sort_values('week', ascending=False)['week'])[0]

Join `train` к `facts`

In [ ]:
df = pd.merge(
    facts,
    train,
    on=['store_id', 'calendar_dt'],
    how='right',
    suffixes=('_facts', '_train')
    )

Очистка пропусков

# TRAIN TEST SPLIT

In [ ]:
last_week_passed = list(df.sort_values('weeks_past', ascending=False)['weeks_past'])[0]
train_data = df[df['weeks_past'] < 0.8 * last_week_passed]
val_data = df[
    (df['weeks_past'] >= 0.8 * last_week_passed) &
    (df['weeks_past'] < 0.9 * last_week_passed)
    ]
test_data = df[df['weeks_past'] >= 0.9 * last_week_passed]

In [ ]:
columns =  [
    'fact_staff_value_lag_1',
    'fact_load_factor_lag_1',
    'num_available_couriers_lag_1',
    'fact_num_orders_lag_1',
    'fact_percent_lateness_lag_1',
    'city_nm',
    'store_lifetime_in_days',
    'fact_staff_churn',
    'flag_high_load_lag_1','marketing_costs_lag_1',
    'fact_couriers_with_shifts_lag_1',
    'week_facts',
    'year_facts'
    ]

X_train =  train_data[columns]
y_train = train_data['target']
X_val =  val_data[columns]
y_val = val_data['target']
X_test = test_data[columns]
y_test = test_data['target']


# Train Boostings

In [ ]:
model_catboost = CatBoostRegressor(
    learning_rate=0.15,
    depth=3,
    eval_metric='MAE',
    verbose=False
    )

model_catboost.fit(
    X_train,
    y=y_train,
    cat_features=['city_nm'],
    use_best_model=True,
    eval_set=(X_val, y_val))

Проверка WAPE

In [ ]:
def WAPE(y, y_pred):
    return np.sum(np.abs(y - y_pred)) / np.sum(y)

In [ ]:
y_pred = model_catboost.predict(X_test)
wape = WAPE(y_test, y_pred)
print(f"WAPE на валидации: {wape:.4f}")

## Попробую обучить CatBoost с optuna

In [ ]:
cat_features = ['city_nm']

def objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.1, 0.2, log=False)
    depth         = 3
    iterations    = 50
    l2_leaf_reg   = 1e-2

    model = CatBoostRegressor(
        learning_rate=learning_rate,
        depth=depth,
        iterations=iterations,
        l2_leaf_reg=l2_leaf_reg,
        cat_features=cat_features,
        eval_metric='MAE',
        verbose=False,
        random_seed=42
    )

    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        use_best_model=True
    )

    y_pred = model.predict(X_val)
    denominator = np.sum(y_val)
    wape = np.sum(np.abs(y_val - y_pred)) / denominator

    return wape

print("Запуск Optuna...")
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Лучший trial:")
print(f"   WAPE: {study.best_trial.value:.4f}")
print("   Параметры:")
for key, value in study.best_trial.params.items():
    print(f"     {key}: {value}")

In [ ]:
best_params = study.best_trial.params
model_catboost = CatBoostRegressor(
    **best_params,
    cat_features=cat_features,
    verbose=False,
    random_seed=42
)

model_catboost.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

In [ ]:

y_pred = model_catboost.predict(X_test)
wape = WAPE(y_test, y_pred)
print(f"FINAL WAPE = {wape}")